# Skripsi: Integrasi Confident Learning dan CORN pada IndoBERT
Notebook final Ryan Besto Saragih (NIM 23051204205)
Pembimbing: Anita Qoiriah, S.Kom., M.Kom.

Pipeline: scraping → kunci korpus (SHA-256) → praproses → split 70:10:20 grouped → ablasi proxy P1–P4
(+ kalibrasi, ECE, sensitivitas K) → deteksi noise → validasi manusia → 6 skenario × 3 seed →
uji signifikansi (**5 hipotesis pre-registered, H1–H5**) → gold test → **sensitivitas split (3 partisi)**.

Aturan: jalankan berurutan. Cell scraping/split/proxy otomatis dilewati bila hasil sudah ada.
**Jangan hapus** `proxy_cache/`, `human_validation/`, `gold_test/`, `all_reviews_master.csv`.
Split utama = seed 42. Validasi manusia dan gold test hanya berlaku untuk split utama.

In [ ]:
# ==========================================================
# CELL 2: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/corn-cl-indobert-rating-noise"
REPO_DIR = "/content/corn-cl-indobert-rating-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

## Instalasi Dependencies

Termasuk `py3langid` untuk filter bahasa saat scraping. Pastikan `requirements.txt`
di repo sudah memuat: `pandas`, `numpy`, `scikit-learn`, `torch`, `transformers`,
`cleanlab`, `coral-pytorch`, `statsmodels`, `scipy`, `google-play-scraper`,
`py3langid`.

In [ ]:
# ==========================================================
# CELL 4: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pip", "show", "coral-pytorch"], capture_output=True, text=True).stdout[:200])

## Import Umum
Dipakai berulang di banyak cell berikutnya — diletakkan sekali di sini supaya
tidak diulang-ulang di tiap cell.

In [ ]:
# ==========================================================
# CELL 6: IMPORT UMUM
# ==========================================================
import os
import pandas as pd
import numpy as np
import platform, torch, transformers, cleanlab, coral_pytorch, sklearn, scipy, statsmodels

from src import config
print(f"📌 Proxy default aktif: [{config.PROXY_ID}] {config.PROXY_NAME}")
print(f"📁 Drive root: {config.DRIVE_ROOT}")

from packaging.version import Version
assert Version(coral_pytorch.__version__) >= Version("1.3.0"), "coral-pytorch < 1.3.0: normalisasi corn_loss berbeda dari paper!"
env = (f"python={platform.python_version()}\ntorch={torch.__version__}\ntransformers={transformers.__version__}\n"
       f"cleanlab={cleanlab.__version__}\ncoral_pytorch={coral_pytorch.__version__}\nsklearn={sklearn.__version__}\n"
       f"scipy={scipy.__version__}\nstatsmodels={statsmodels.__version__}\n"
       f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
print(env)
open(os.path.join(config.PRIMARY_RESULTS_DIR, "environment.txt"), "w").write(env)

## Scraping Google Play Store

Scraping dengan filter bahasa Indonesia inline (`py3langid`), target per rating
sudah dinaikkan untuk mengantisipasi susut data sepanjang pipeline (praproses →
split → K-Fold → pruning), dan dilengkapi laporan kualitas data (distribusi
tanggal per app/rating untuk cek confound temporal, distribusi panjang teks,
deteksi teks duplikat) yang otomatis tersimpan di `results/` untuk dilampirkan
di Bab III dan Bab IV.

⚠️ **JALANKAN SEKALI SAJA SEUMUR PROYEK.** Setelah selesai, `all_reviews_master.csv`
dikunci dan dipakai sebagai sumber kebenaran tunggal. Cell ini **otomatis dilewati**
kalau file tersebut sudah ada — hapus manual hanya kalau memang sengaja ingin
scraping ulang dari nol (dan sadar semua eksperimen berikutnya perlu diulang).

Estimasi waktu: bisa 1-3 jam tergantung koneksi dan seberapa jauh Google Play
Store perlu ditelusuri untuk rating 2/3 yang jarang muncul.

In [ ]:
# ==========================================================
# CELL 8: SCRAPING GOOGLE PLAY STORE (AUTO-SKIP KALAU SUDAH ADA)
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
    df_check = pd.read_csv(config.RAW_DATA_FILE)
    print(f"   Baris tersedia: {len(df_check)}")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

## Kunci Korpus + EDA Awal (SEBELUM Praproses)
Dijalankan sekali di atas korpus mentah: dedup review_id, kunci sumber
kebenaran tunggal (manifest + SHA-256, Batasan Masalah butir 7), dan
seluruh angka deskripsi data Subbab III.B (filter bahasa, distribusi
tanggal, panjang teks, duplikat, konflik teks-rating, cek overlap split).

In [ ]:
# ==========================================================
# CELL 9: KUNCI KORPUS + EDA AWAL
# ==========================================================
from scripts.eda_dataset import main as run_eda
run_eda()

In [ ]:
from scripts.eda_dataset import verify_manifest
verify_manifest()   # harus COCOK; kalau tidak, hentikan

## Praproses Teks

Case folding, penghapusan URL/username, translasi emoji ke sentimen (kamus
diperluas), normalisasi elongasi huruf, normalisasi slang (Kamus Alay + kamus
domain, dengan perbaikan pencocokan token bertanda-baca), resolusi konflik
teks-identik-rating-berbeda (voting mayoritas), dan penghapusan duplikat murni.

Preprocessing dijaga minimal (tanpa stemming/stopword removal) sesuai rekomendasi
untuk model berbasis BERT (Wilie dkk. 2020; Koto dkk. 2020).

⚠️ JALANKAN SEKALI SAJA. Split train/val/test yang dihasilkan
(split_train_raw.csv / split_val.csv / split_test.csv) dikunci dan dipakai
di seluruh eksperimen berikutnya, termasuk ablasi proxy P1-P4. Cell ini
otomatis dilewati kalau ketiga split sudah ada.

In [ ]:
# ==========================================================
# CELL 10: PREPROCESSING + SPLIT 70:10:20 GROUPED-STRATIFIED
# ==========================================================
from src.preprocess import run_preprocessing
from src.data_split import create_splits
from src import config

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.VAL_FILE) and os.path.exists(config.TEST_FILE):
    print("✅ Split train/val/test sudah ada -- dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_val = pd.read_csv(config.VAL_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
else:
    print("=" * 60); print(" PREPROCESSING "); print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT 70:10:20 (grouped berdasarkan teks, stratified rating, seed=42) ")
    print("=" * 60)
    df_train, df_val, df_test = create_splits(config.CLEAN_TEXT_FILE, seed=42)

In [ ]:
# ==========================================================
# CELL 10B: SAMPLING SUBSET UJI EMAS (GOLD TEST SUBSET) -- Subbab 3.9.4
# ==========================================================
from src import config, gold_test

if os.path.exists(config.GOLD_TEST_SAMPLE_FILE):
    print(f"✅ {config.GOLD_TEST_SAMPLE_FILE} sudah ada -- sampling dilewati.")
else:
    gold_test.sample_gold_test_subset(seed=42)
    gold_test.export_second_annotator_gold_subset()

print("\n⚠️  Anotasi gold test bisa dikerjakan PARALEL sambil menunggu training")
print("   6 skenario (Cell 20) berjalan -- evaluasinya baru dijalankan di Cell 23")
print("   setelah training tuntas. Isi 'human_gold_rating' (1-5) langsung di file di atas.")

gold_test.write_rubric()

In [ ]:
tr, va, te = [pd.read_csv(p) for p in (config.TRAIN_RAW_FILE, config.VAL_FILE, config.TEST_FILE)]
T = lambda d: set(d["cleaned_text"])
print("overlap train∩val, train∩test, val∩test:", len(T(tr)&T(va)), len(T(tr)&T(te)), len(T(va)&T(te)))  # harus 0 0 0
print("Bandingkan: simulasi split acak non-grouped di EDA -> 16,58% teks uji muncul di train.")

## Ablasi Proxy Classifier (P1-P4) -- Subbab III.F.1

Menjalankan Confident Learning untuk empat tahap ablasi proxy classifier
(Tabel 3.1, Bab III): P1 (CLS embedding beku + LR), P2 (mean-pooling beku
+ LR), P3 (IndoBERT fine-tuned, Cross-Entropy), P4 (IndoBERT fine-tuned,
CORN -- proxy final). Sesuai Batasan Masalah, hanya empat proxy ini yang
dipakai -- tidak ada tahap tambahan di luar backbone IndoBERT tunggal.

Setiap proxy: (a) memakai skema K-Fold yang identik (shuffle=True, seed=42)
untuk seluruh tahap, (b) dikalibrasi lewat temperature scaling PER-FOLD
untuk P3/P4 (T dicari dari validasi internal dalam bagian latih tiap fold,
bukan dari OOF gabungan -- Subbab III.F butir 2), (c) metode filter cleanlab
dipilih otomatis lewat kriteria eksplisit (paling dekat dengan estimasi
teoretis num_label_issues() versi off_diagonal_calibrated).

Hasilnya terkumpul di results/proxy_ablation_table.csv -- inilah Tabel 3.1
di Bab III/IV proposal.

⏱️ P3 dan P4 butuh fine-tuning K-Fold (5 fold × 3 epoch masing-masing) --
cell ini bisa memakan waktu cukup lama.

In [ ]:
# ==========================================================
# CELL 12: PILOT STUDY -- ABLASI PROXY P1-P4
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

PILOT_PROXY_IDS = [0, 1, 2, 3]

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study (P1-P4) selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table)

## Kunci Proxy Final (P4) + Ekspor Sample Validasi Manusia

Menjalankan ulang Confident Learning dengan proxy final terkunci (P4/CORN,
sesuai skema integrasi ketat Bab I.A) -- akan memuat dari cache kalau sudah
pernah dihitung di Cell 12, jadi cepat. Ini menghasilkan sample 100 baris
untuk validasi manusia, dengan alokasi TIDAK PROPORSIONAL (Subbab III.I.3):
minimal 15 baris per bin jarak ordinal >=2 (titik keputusan ambang
severity-aware pruning), sisa dialokasikan ke bin jarak 1.

⚠️ BERHENTI setelah cell ini -- isi dulu human_verdict secara manual
sebelum lanjut ke Cell 17.

In [ ]:
# ==========================================================
# CELL 16: KUNCI PROXY FINAL (P4) -- WAJIB SEBELUM VALIDASI MANUSIA
# ==========================================================
from src import config

config.set_proxy(3)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

print("ℹ️  Resolusi konflik teks-identik-rating-beda sekarang otomatis dieksekusi")
print("   di dalam run_confident_learning() SETELAH deteksi CL (Subbab III.D butir 8).")

print("\n⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 17 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan Cell 17 untuk cek kelengkapan + hitung agreement rate.")

## Hitung Agreement Rate Validasi Manusia

Menjalankan **setelah** kamu selesai mengisi `human_verdict` secara manual di
file yang disebut Cell 16. Melaporkan agreement rate keseluruhan dan per-bin
`rating_diff` — breakdown per-bin ini penting sebagai bukti kualitatif
pendukung/penolak asumsi ambang severity-aware pruning (Subbab 3.7).

In [ ]:
from src import config
import pandas as pd

df_check = pd.read_csv(config.HUMAN_VALIDATION_FILE, sep=None, engine="python")
print(df_check.shape)
print(df_check["human_verdict"].value_counts(dropna=False))

In [ ]:
# ==========================================================
# CELL 18: HITUNG AGREEMENT VALIDASI MANUSIA
# ==========================================================
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 19 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

## Anotator Kedua / Test-Retest -- Reliabilitas Validasi Noise (Subbab III.I.3)

Kalau tersedia penilai kedua, jalankan `export_second_annotator_subset()`
lalu minta penilai kedua mengisi verdict secara buta (sama seperti anotator
pertama), lalu hitung Cohen's kappa. Kalau hanya ada satu penilai, gunakan
jalur fallback test-retest (menilai ulang ~10% sampel secara mandiri, tanpa
melihat verdict sebelumnya).

Langkah ini penting untuk laporan Bab IV: Cohen's kappa (atau test-retest
kappa) menjadi bukti kuantitatif langsung soal subjektivitas domain rating
ini, relevan untuk rumusan masalah butir 4.

In [ ]:
# ==========================================================
# CELL 18B: ANOTATOR KEDUA + COHEN'S KAPPA -- VALIDASI NOISE (Subbab III.I.3)
# ==========================================================
from src import human_validation as hv

hv.export_second_annotator_subset()
print("\n⚠️  Minta penilai kedua mengisi 'human_verdict' pada file di atas, BUTA seperti anotator pertama.")
print("Setelah selesai, jalankan:")
print(">>> from src import human_validation as hv")
print(">>> hv.compute_interannotator_kappa()")
print("\nKalau hanya ada satu penilai (fallback single-annotator), jalankan sebagai gantinya:")
print(">>> hv.export_retest_subset()")
print(">>> # isi manual, lalu:")
print(">>> hv.compute_test_retest_reliability()")

In [ ]:
# ==========================================================
# CELL 18C: HITUNG RELIABILITAS (jalankan setelah anotasi kedua/retest selesai)
# ==========================================================
from src import human_validation as hv

# Pilih SATU sesuai jalur yang dipakai:
kappa = hv.compute_interannotator_kappa()      # kalau ada penilai kedua
# kappa = hv.compute_test_retest_reliability()  # kalau fallback single-annotator

In [ ]:
# ==========================================================
# CELL 19B (SISIPAN): DEFINISI ULANG SCENARIOS (tanpa training)
# Diperlukan karena runtime baru -- variabel 'scenarios' dari Cell 20
# tidak persist, tapi checkpoint hasil training-nya SUDAH ada di Drive.
# ==========================================================
from src import config

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_RESOLVED_FILE, "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE, "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE, "loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_RESOLVED_FILE, "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE, "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE, "loss": "corn"},
]
print("✅ scenarios didefinisikan ulang (checkpoint diambil dari Drive, tidak training ulang).")

## Training 6 Skenario × 3 Seed

Melatih 6 skenario (kombinasi 3 varian data × 2 fungsi loss): M1-M3 (raw/
hard-prune/severity-prune × Cross-Entropy), M4-M6 (raw/hard-prune/severity-prune
× CORN). Setiap skenario dilatih pada 3 seed (42, 123, 2024) dengan mixed-precision
training, linear warmup 10% dari total langkah, dan early stopping
(berhenti kalau Val MAE tidak membaik 3 epoch berturut-turut, patience=3).
Checkpoint terbaik dipilih berdasarkan QWK validasi (metrik keputusan
utama, Tabel 3.3), dengan MAE validasi sebagai tie-breaker apabila QWK
antar-checkpoint identik.

Val dan test dimuat dari partisi TETAP (config.VAL_FILE / config.TEST_FILE),
sama untuk seluruh 18 kombinasi skenario × seed -- bukan di-split ulang per
run -- sesuai Subbab III.E.

Auto-resume: kalau sesi Colab terputus, jalankan ulang cell ini — training
akan melanjutkan dari skenario/seed terakhir yang belum selesai, bukan mulai
dari nol.

⏱️ Ini cell paling lama di seluruh notebook — bisa beberapa jam tergantung GPU
yang dialokasikan Colab.

In [ ]:
# ==========================================================
# CELL 20: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

print(f"🔧 Training M1-M6 akan pakai backbone: {config.PRETRAINED_MODEL_NAME} | "
      f"proxy final: {config.PROXY_NAME}")

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_RESOLVED_FILE, "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE, "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE, "loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_RESOLVED_FILE, "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE, "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE, "loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(" 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆")
print("=" * 100)
display(df_final)

## Uji Signifikansi — Wilcoxon + Holm-Bonferroni (5 Hipotesis) + CI Bootstrap QWK
H1: CORN vs CE pada data mentah (M4 vs M1)
H2: severity-aware vs baseline pada CORN (M6 vs M4)
H3: hard-prune vs baseline pada CORN (M5 vs M4)
H4: severity-aware vs hard-prune langsung (M6 vs M5)
H5: severity-aware vs baseline pada CE (M3 vs M1) — apakah manfaat cleaning
    selektif spesifik pada arsitektur ordinal, atau berlaku umum lintas fungsi loss
QWK adalah metrik keputusan utama tapi tidak bisa diuji Wilcoxon (dihitung
atas matriks konfusi agregat) — dilaporkan via CI bootstrap terpisah (III.I.2).

In [ ]:
# ==========================================================
# CELL 22: UJI SIGNIFIKANSI -- WILCOXON+HOLM-BONFERRONI (5 HIPOTESIS) + CI QWK
# ==========================================================
from src.significance import (
    collect_all_predictions, aggregate_errors_across_seeds,
    run_significance_test, run_all_effect_sizes, run_all_qwk_effect_sizes,
)
from src import config

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Wilcoxon Signed-Rank (H1-H5) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size MAE + CI 95% (pelengkap Wilcoxon)...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)
effect_sizes.to_csv(os.path.join(config.RESULTS_DIR, "effect_sizes.csv"), index=False)

print("\n📐 Selisih QWK + CI 95% bootstrap (metrik keputusan utama, TANPA Wilcoxon)...")
qwk_effect_sizes = run_all_qwk_effect_sizes(true_labels, preds_per_seed)
display(qwk_effect_sizes)
qwk_effect_sizes.to_csv(os.path.join(config.RESULTS_DIR, "qwk_effect_sizes.csv"), index=False)

## Evaluasi Subset Uji Emas (Gold Test Subset) -- Pemeriksaan Ketahanan (Subbab III.I.4)

Mengevaluasi keenam skenario pada subset uji emas yang telah diverifikasi
manusia (Cell 10B), sebagai pemeriksaan ketahanan SEKUNDER terhadap hasil
H1-H5 pada data uji penuh (Cell 22). Baris dengan opsi "tidak dapat
ditentukan dari teks" (ND) dikeluarkan dari perhitungan metrik.

⚠️ Pastikan kolom `human_gold_rating` di
`gold_test/gold_test_sample.csv` sudah terisi lengkap (1-5, atau ND untuk
kasus yang bergantung info eksternal) sebelum menjalankan cell ini.

In [ ]:
# ==========================================================
# CELL 23: EVALUASI 6 SKENARIO PADA SUBSET UJI EMAS -- Subbab III.I.4
# ==========================================================
from src import config, gold_test

result = gold_test.evaluate_on_gold_subset(scenarios)
if result is not None:
    df_gold_result, df_gold_effect = result
    print("\n✅ Evaluasi subset uji emas selesai -- lampirkan ke Bab IV berdampingan "
          "dengan hasil data uji penuh (Cell 20 & 22).")
    print("\n📌 Bandingkan arah keunggulan di atas dengan hasil H1-H5 pada data uji penuh:")
    print("   konsisten memperkuat validitas, tidak konsisten dilaporkan sebagai temuan")
    print("   yang membatasi generalisasi (Subbab III.I.4).")
    gold_test.compute_gold_interannotator_kappa()
else:
    print(f"\n⛔ Lengkapi dulu {config.GOLD_TEST_SAMPLE_FILE} sebelum menjalankan cell ini.")

## Ringkasan Akhir — Siap Disalin ke Bab 1 (Pilot Study) dan Bab 4

Semua dibaca dari file di Drive (bukan variabel session), jadi cell ini bisa
dijalankan kapan saja, tidak peduli cell mana yang sudah/belum jalan di sesi
Colab saat ini.

In [ ]:
# ==========================================================
# CELL 24: RINGKASAN AKHIR
# ==========================================================
import os
import pandas as pd
from src import config

print("=" * 70)
print(" RINGKASAN LENGKAP ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (P1-P4) -- untuk Bab III/IV:")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    display(pd.read_csv(config.PROXY_QUALITY_LOG_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 12 dulu.")

print("\n[2] LAPORAN KUALITAS DATA SCRAPING -- untuk Bab III:")
for fname in ["scraping_summary.csv", "date_distribution_per_app_rating.csv",
              "text_length_summary.csv", "duplicate_review_texts.csv"]:
    fpath = os.path.join(config.RESULTS_DIR, fname)
    status = "✅" if os.path.exists(fpath) else "⚠️ belum ada"
    print(f"   {status} {fpath}")

print("\n[2B] EDA + PENGUNCIAN KORPUS + PREPROCESSING -- untuk Bab III:")
for fname in ["dataset_manifest.json", "split_summary.csv",
              "preprocessing_summary.csv", "token_truncation_summary.csv"]:
    fpath = os.path.join(config.RESULTS_DIR, fname)
    status = "✅" if os.path.exists(fpath) else "⚠️ belum ada"
    print(f"   {status} {fpath}")
eda_dir = os.path.join(config.RESULTS_DIR, "eda")
status = "✅" if os.path.exists(eda_dir) else "⚠️ belum ada"
print(f"   {status} {eda_dir}/ (tabel & gambar EDA lengkap III.B, III.D, III.E)")

print("\n[3] VALIDASI MANUSIA (deteksi noise) -- untuk Bab IV:")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 18 dulu.")

kappa_file = config.HUMAN_VALIDATION_KAPPA_FILE
status = "✅" if os.path.exists(kappa_file) else "⚠️ belum ada -- jalankan Cell 18B/18C"
print(f"   {status} {kappa_file}")

print("\n[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed, data uji penuh) -- untuk Bab IV:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 20 dulu.")

print("\n[5] UJI SIGNIFIKANSI (5 hipotesis pre-registered, H1-H5) -- untuk Bab IV:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n[6] EFFECT SIZE MAE + CI 95% -- untuk Bab IV:")
effect_size_file = os.path.join(config.RESULTS_DIR, "effect_sizes.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n[7] EFFECT SIZE QWK + CI 95% (metrik keputusan utama) -- untuk Bab IV:")
qwk_file = os.path.join(config.RESULTS_DIR, "qwk_effect_sizes.csv")
if os.path.exists(qwk_file):
    display(pd.read_csv(qwk_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n[8] EVALUASI SUBSET UJI EMAS (pemeriksaan ketahanan, Subbab III.I.4) -- untuk Bab IV:")
if os.path.exists(config.GOLD_TEST_EVAL_FILE):
    display(pd.read_csv(config.GOLD_TEST_EVAL_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 23 dulu.")

print("\n[9] ARAH & EFFECT SIZE SUBSET UJI EMAS -- untuk Bab IV:")
if os.path.exists(config.GOLD_TEST_EFFECT_SIZE_FILE):
    display(pd.read_csv(config.GOLD_TEST_EFFECT_SIZE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 23 dulu.")

und_file = os.path.join(config.RESULTS_DIR, "gold_test_undetermined_summary.csv")
status = "✅" if os.path.exists(und_file) else "⚠️ belum ada"
print(f"\n   {status} {und_file} (proporsi 'tidak dapat ditentukan dari teks')")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)